# Assignment 6: Medians and Order Statistics & Elementary Data Structures

- **Part 1:** Deterministic selection (Median of Medians) and randomized selection (Randomized Quickselect)
- **Part 1B:** Empirical benchmarking for the selection algorithms
- **Part 2:** Arrays, matrices, stacks, queues, linked lists, and an optional rooted tree


In [11]:
"""
Part 1
------
1. Deterministic selection using the Median of Medians method
2. Randomized selection using Randomized Quickselect
3. A small benchmark utility for empirical comparison
"""

from __future__ import annotations

import argparse
import csv
import random
import statistics
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterator, List, Optional, Sequence, Tuple



## Part 1: Selection Algorithms

This section implements:
- **Deterministic Select** using the **Median of Medians** method
- **Randomized Select** using **Randomized Quickselect**

Both functions return the **k-th smallest element** using **1-based indexing** for `k`.

In [12]:
# PART 1: SELECTION ALGORITHMS

class SelectionError(ValueError):
    """Custom error used for invalid selection inputs."""


def validate_selection_input(arr: Sequence[int], k: int) -> None:
    """
    Check that the input array and k value are valid.
    We keep this logic in one helper function so both selection algorithms
    behave the same way for bad input.
    """
    if arr is None:
        raise SelectionError("Input array must not be None.")
    if len(arr) == 0:
        raise SelectionError("Input array must not be empty.")
    if not 1 <= k <= len(arr):
        raise SelectionError(f"k must be between 1 and {len(arr)} inclusive.")


def partition_three_way(values: List[int], pivot: int) -> Tuple[List[int], List[int], List[int]]:
    """
    Partition the values into three lists relative to the pivot.
    A 3-way partition cleanly separates:
        - values smaller than pivot
        - values equal to pivot
        - values larger than pivot

    This makes duplicate-heavy inputs easier and safer to handle.
    """
    less: List[int] = []
    equal: List[int] = []
    greater: List[int] = []

    for value in values:
        if value < pivot:
            less.append(value)
        elif value > pivot:
            greater.append(value)
        else:
            equal.append(value)

    return less, equal, greater


def median_of_small_group(group: List[int]) -> int:
    """
    Return the median of a small group.
    In Median of Medians, the array is broken into groups of 5. Since each group
    is tiny, sorting it directly is perfectly fine and keeps the code simple.
    """
    group.sort()
    return group[len(group) // 2]


def deterministic_select(arr: Sequence[int], k: int) -> int:
    """
    Return the k-th smallest element using Median of Medians.

    arr : input sequence
    k   : 1-based position (k=1 means the smallest element)

    This algorithm guarantees O(n) running time in the worst case because it
    chooses a pivot carefully rather than randomly.
    """
    validate_selection_input(arr, k)

    # Make a copy so the original input is not modified.
    values = list(arr)

    # Internally we use 0-based indexing, so convert k here once.
    target_index = k - 1
    return _deterministic_select_recursive(values, target_index)


def _deterministic_select_recursive(values: List[int], target_index: int) -> int:
    """
    Recursive helper for deterministic selection.

    target_index is 0-based within the current 'values' list.
    """
    n = len(values)

    # Base case:
    # For very small lists, just sort and directly return the target element.
    # This is simpler than recursing further and does not change asymptotic cost.
    if n <= 5:
        values.sort()
        return values[target_index]

    # Step 1: split the array into groups of at most 5 elements.
    groups = [values[i:i + 5] for i in range(0, n, 5)]

    # Step 2: find the median of each small group.
    medians = [median_of_small_group(group) for group in groups]

    # Step 3: recursively find the median of the medians.
    # This chosen pivot is what gives the algorithm its worst-case O(n) guarantee.
    pivot = _deterministic_select_recursive(medians, len(medians) // 2)

    # Step 4: partition around the pivot.
    less, equal, greater = partition_three_way(values, pivot)

    # Step 5: decide which side contains the target index.
    if target_index < len(less):
        # The k-th smallest element is in the smaller side.
        return _deterministic_select_recursive(less, target_index)

    if target_index < len(less) + len(equal):
        # The target falls inside the "equal to pivot" block.
        # That means the answer is just the pivot itself.
        return pivot

    # Otherwise the target is in the larger side.
    # We subtract off the number of elements we skipped.
    new_index = target_index - len(less) - len(equal)
    return _deterministic_select_recursive(greater, new_index)


def randomized_select(arr: Sequence[int], k: int, rng: random.Random | None = None) -> int:
    """
    Return the k-th smallest element using Randomized Quickselect.

    This algorithm has expected O(n) running time, which is very good in practice.
    However, unlike Median of Medians, its worst case is O(n^2) if unlucky pivots
    are chosen repeatedly.
    """
    validate_selection_input(arr, k)

    values = list(arr)
    target_index = k - 1

    # Allow a custom random generator for reproducible experiments.
    generator = rng if rng is not None else random
    return _randomized_select_iterative(values, target_index, generator)


def _randomized_select_iterative(values: List[int], target_index: int, rng: random.Random) -> int:
    """
    Iterative Quickselect helper.

    We use a loop instead of deep recursion to keep the control flow simple and
    reduce recursion overhead in practice.
    """
    while True:
        # Small input shortcut.
        if len(values) <= 5:
            values.sort()
            return values[target_index]

        # Pick a random pivot.
        pivot = values[rng.randrange(len(values))]

        # Partition the data around that pivot.
        less, equal, greater = partition_three_way(values, pivot)

        if target_index < len(less):
            # The target lies in the smaller values.
            values = less
        elif target_index < len(less) + len(equal):
            # The target lies in the equal block, so the pivot is the answer.
            return pivot
        else:
            # Move into the greater side and adjust the target index.
            target_index -= len(less) + len(equal)
            values = greater


def verify_selection_against_sort(arr: Sequence[int], k: int) -> bool:
    """
    Simple correctness check.

    We compare both selection algorithms against Python's sorted() result.
    This is useful for quick validation and small demo tests.
    """
    expected = sorted(arr)[k - 1]
    det_result = deterministic_select(arr, k)
    rand_result = randomized_select(arr, k)
    return det_result == expected and rand_result == expected





## Part 1B: Empirical Benchmarking

This section creates test arrays with different input distributions and measures the average runtime of each selection method.

Included distributions:
- random
- sorted
- reverse-sorted
- many duplicates

In [16]:
# PART 1B: EMPIRICAL ANALYSIS / BENCHMARKING

def make_input_array(size: int, distribution: str) -> List[int]:
    """
    Create an input array of a requested distribution.
    """
    if distribution == "random":
        values = list(range(size))
        random.shuffle(values)
        return values

    if distribution == "sorted":
        return list(range(size))

    if distribution == "reverse_sorted":
        return list(range(size, 0, -1))

    if distribution == "many_duplicates":
        return [random.randint(0, max(1, size // 10)) for _ in range(size)]

    raise ValueError(f"Unknown distribution: {distribution}")


def time_algorithm(func: Callable[[List[int], int], int], arr: List[int], k: int, trials: int = 5) -> float:
    """
    Measure the average running time of a selection function.
    """
    times: List[float] = []

    for _ in range(trials):
        data = arr[:]

        start = time.perf_counter()
        func(data, k)
        end = time.perf_counter()

        times.append(end - start)

    return statistics.mean(times)


def run_selection_benchmarks(
    output_file: str = "selection_benchmark_results.csv",
    sizes: Optional[List[int]] = None,
    trials: int = 5,
) -> None:
    """
    Run benchmark experiments for the two selection algorithms.
    Results are saved as a CSV file.
    """
    if sizes is None:
        sizes = [1000, 5000, 10000, 20000]

    distributions = ["random", "sorted", "reverse_sorted", "many_duplicates"]
    rows: List[dict[str, Any]] = []

    print("Running selection benchmarks...\n")

    for size in sizes:
        for distribution in distributions:
            arr = make_input_array(size, distribution)
            k = size // 2 if size > 0 else 0

            det_time = time_algorithm(deterministic_select, arr, k, trials=trials)
            rand_time = time_algorithm(randomized_select, arr, k, trials=trials)

            row = {
                "size": size,
                "distribution": distribution,
                "k": k,
                "deterministic_seconds": det_time,
                "randomized_seconds": rand_time,
            }
            rows.append(row)
            print(row)

    output_path = Path.cwd() / output_file

    with output_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "size",
                "distribution",
                "k",
                "deterministic_seconds",
                "randomized_seconds",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nCSV saved successfully at:\n{output_path}")


# Actually run the benchmark
if __name__ == "__main__":
    run_selection_benchmarks()

Running selection benchmarks...

{'size': 1000, 'distribution': 'random', 'k': 500, 'deterministic_seconds': 0.0005135679999511922, 'randomized_seconds': 0.00023406120017170905}
{'size': 1000, 'distribution': 'sorted', 'k': 500, 'deterministic_seconds': 0.00034688740015553776, 'randomized_seconds': 0.00024273120016005123}
{'size': 1000, 'distribution': 'reverse_sorted', 'k': 500, 'deterministic_seconds': 0.0003794474001551862, 'randomized_seconds': 0.0001809146004234208}
{'size': 1000, 'distribution': 'many_duplicates', 'k': 500, 'deterministic_seconds': 0.00024147539988916833, 'randomized_seconds': 0.00027814440018119057}
{'size': 5000, 'distribution': 'random', 'k': 2500, 'deterministic_seconds': 0.0026928264001981005, 'randomized_seconds': 0.0009172292000585002}
{'size': 5000, 'distribution': 'sorted', 'k': 2500, 'deterministic_seconds': 0.001835564599969075, 'randomized_seconds': 0.0007199231997219613}
{'size': 5000, 'distribution': 'reverse_sorted', 'k': 2500, 'deterministic_secon

## Part 2: Elementary Data Structures

This section implements the required data structures from scratch:
- dynamic array
- matrix
- stack
- queue
- singly linked list
- optional rooted tree

The code comments explain the key design choices and time-complexity trade-offs.

In [17]:
# PART 2: ELEMENTARY DATA STRUCTURES

class ArrayList:
    """
    Python already has built-in lists, but this class shows how a dynamic array
    works internally: a fixed-capacity storage area is resized when needed.

    Operations implemented:
        - append
        - insert
        - delete
        - get
        - set
        - to_list
    """

    def __init__(self, capacity: int = 4) -> None:
        # Start with a small positive capacity.
        self._capacity = max(1, capacity)
        self._size = 0
        self._data: List[Optional[Any]] = [None] * self._capacity

    def __len__(self) -> int:
        return self._size

    def __repr__(self) -> str:
        return f"ArrayList({self.to_list()})"

    def _resize(self, new_capacity: int) -> None:
        """
        Allocate a larger (or smaller) storage array and copy existing values.

        Resizing costs O(n), but it does not happen on every append. That is why
        append is O(1) amortized rather than O(1) worst-case every time.
        """
        new_data: List[Optional[Any]] = [None] * new_capacity
        for i in range(self._size):
            new_data[i] = self._data[i]
        self._data = new_data
        self._capacity = new_capacity

    def append(self, value: Any) -> None:
        """Add a value at the end."""
        if self._size == self._capacity:
            self._resize(self._capacity * 2)

        self._data[self._size] = value
        self._size += 1

    def insert(self, index: int, value: Any) -> None:
        """
        Insert a value at a given position.

        All later elements must shift one step to make room.
        """
        if not 0 <= index <= self._size:
            raise IndexError("Index out of range.")

        if self._size == self._capacity:
            self._resize(self._capacity * 2)

        # Shift elements one position to the right.
        for i in range(self._size, index, -1):
            self._data[i] = self._data[i - 1]

        self._data[index] = value
        self._size += 1

    def delete(self, index: int) -> Any:
        """
        Remove and return the value at index.

        Deletion from the middle also takes O(n) because the remaining elements
        must shift left to fill the gap.
        """
        if not 0 <= index < self._size:
            raise IndexError("Index out of range.")

        removed = self._data[index]

        # Shift elements left to close the gap.
        for i in range(index, self._size - 1):
            self._data[i] = self._data[i + 1]

        self._data[self._size - 1] = None
        self._size -= 1

        # Optional shrink step to avoid wasting too much memory.
        if 0 < self._size <= self._capacity // 4 and self._capacity > 4:
            self._resize(max(4, self._capacity // 2))

        return removed

    def get(self, index: int) -> Any:
        """Return the value at index."""
        if not 0 <= index < self._size:
            raise IndexError("Index out of range.")
        return self._data[index]

    def set(self, index: int, value: Any) -> None:
        """Replace the value at index."""
        if not 0 <= index < self._size:
            raise IndexError("Index out of range.")
        self._data[index] = value

    def to_list(self) -> List[Any]:
        """Return the used part of the array as a normal Python list."""
        return [self._data[i] for i in range(self._size)]


class Matrix:
    """
    A simple matrix represented as a list of lists.

    This is not a full linear algebra package. It only provides basic operations
    needed for this assignment: access, update, row insertion, and row deletion.
    """

    def __init__(self, rows: int, cols: int, fill: Any = 0) -> None:
        if rows <= 0 or cols <= 0:
            raise ValueError("rows and cols must be positive")

        self.rows = rows
        self.cols = cols

        # Create the 2D structure row by row.
        self.data = [[fill for _ in range(cols)] for _ in range(rows)]

    def get(self, row: int, col: int) -> Any:
        return self.data[row][col]

    def set(self, row: int, col: int, value: Any) -> None:
        self.data[row][col] = value

    def insert_row(self, values: List[Any]) -> None:
        """
        Append a new row to the matrix.

        The new row must have the correct number of columns.
        """
        if len(values) != self.cols:
            raise ValueError("Inserted row must have the same number of columns.")

        # Copy the row so external changes do not affect matrix internals.
        self.data.append(values[:])
        self.rows += 1

    def delete_row(self, row: int) -> List[Any]:
        """Delete and return one row."""
        removed = self.data.pop(row)
        self.rows -= 1
        return removed

    def __repr__(self) -> str:
        return f"Matrix(rows={self.rows}, cols={self.cols}, data={self.data})"


class ArrayStack:
    """
    Stack implemented using a dynamic array.

    LIFO means: Last In, First Out.
    The last item pushed is the first one popped.
    """

    def __init__(self) -> None:
        self._items: List[Any] = []

    def push(self, value: Any) -> None:
        self._items.append(value)

    def pop(self) -> Any:
        if self.is_empty():
            raise IndexError("Pop from empty stack.")
        return self._items.pop()

    def peek(self) -> Any:
        if self.is_empty():
            raise IndexError("Peek from empty stack.")
        return self._items[-1]

    def is_empty(self) -> bool:
        return len(self._items) == 0

    def __len__(self) -> int:
        return len(self._items)

    def __repr__(self) -> str:
        return f"ArrayStack({self._items})"


class ArrayQueue:
    """
    Queue implemented using a circular array.

    FIFO means: First In, First Out.
    The first item enqueued is the first one dequeued.

    If we always shifted elements left after every dequeue, that would be O(n).
    Circular indexing avoids that and keeps enqueue/dequeue O(1) amortized.
    """

    def __init__(self, capacity: int = 4) -> None:
        self._capacity = max(4, capacity)
        self._data: List[Optional[Any]] = [None] * self._capacity
        self._front = 0
        self._size = 0

    def __len__(self) -> int:
        return self._size

    def __repr__(self) -> str:
        elements = [self._data[(self._front + i) % self._capacity] for i in range(self._size)]
        return f"ArrayQueue({elements})"

    def is_empty(self) -> bool:
        return self._size == 0

    def _resize(self, new_capacity: int) -> None:
        """
        Rebuild the queue into a new underlying array.

        This keeps the front element at index 0 after resizing, which simplifies
        later enqueue/dequeue operations.
        """
        new_data: List[Optional[Any]] = [None] * new_capacity
        for i in range(self._size):
            new_data[i] = self._data[(self._front + i) % self._capacity]

        self._data = new_data
        self._capacity = new_capacity
        self._front = 0

    def enqueue(self, value: Any) -> None:
        if self._size == self._capacity:
            self._resize(self._capacity * 2)

        # Back position is computed relative to front because the queue may wrap.
        back = (self._front + self._size) % self._capacity
        self._data[back] = value
        self._size += 1

    def dequeue(self) -> Any:
        if self.is_empty():
            raise IndexError("Dequeue from empty queue.")

        value = self._data[self._front]
        self._data[self._front] = None

        # Move front one step circularly.
        self._front = (self._front + 1) % self._capacity
        self._size -= 1

        # Shrink when usage becomes small.
        if 0 < self._size <= self._capacity // 4 and self._capacity > 4:
            self._resize(max(4, self._capacity // 2))

        return value

    def front(self) -> Any:
        if self.is_empty():
            raise IndexError("Queue is empty.")
        return self._data[self._front]


@dataclass
class Node:
    """
    Node used in the singly linked list.
    Each node stores a value and a reference to the next node.
    """

    value: Any
    next: Optional["Node"] = None


class SinglyLinkedList:
    """
    A singly linked list.

    Each node stores a value and a reference to the next node.
    Unlike arrays, linked lists do not store items in contiguous memory.
    """

    def __init__(self) -> None:
        self.head: Optional[Node] = None
        self.size = 0

    def __len__(self) -> int:
        return self.size

    def __repr__(self) -> str:
        return f"SinglyLinkedList({self.traverse()})"

    def insert_front(self, value: Any) -> None:
        """
        Insert at the beginning.

        This is O(1) because we only change one pointer.
        """
        self.head = Node(value=value, next=self.head)
        self.size += 1

    def insert_end(self, value: Any) -> None:
        """
        Insert at the end.

        In a singly linked list without a tail pointer, we must walk to the last
        node first, so this is O(n).
        """
        new_node = Node(value=value)

        if self.head is None:
            self.head = new_node
            self.size += 1
            return

        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node
        self.size += 1

    def delete_value(self, value: Any) -> bool:
        """
        Delete the first node containing the given value.

        Returns True if a node was deleted, otherwise False.
        """
        prev: Optional[Node] = None
        current = self.head

        while current is not None:
            if current.value == value:
                if prev is None:
                    # We are deleting the head node.
                    self.head = current.next
                else:
                    prev.next = current.next

                self.size -= 1
                return True

            prev = current
            current = current.next

        return False

    def traverse(self) -> List[Any]:
        """Return all node values from head to end."""
        output: List[Any] = []
        current = self.head

        while current is not None:
            output.append(current.value)
            current = current.next

        return output

    def __iter__(self) -> Iterator[Any]:
        current = self.head
        while current is not None:
            yield current.value
            current = current.next


@dataclass
class TreeNode:
    """Node used in the optional rooted tree."""

    value: Any
    children: List["TreeNode"] = field(default_factory=list)

    def add_child(self, child: "TreeNode") -> None:
        self.children.append(child)


class RootedTree:

    def __init__(self, root_value: Any) -> None:
        self.root = TreeNode(root_value)

    def dfs(self) -> List[Any]:
        """
        Depth-first traversal of the tree.
        This returns the order in which nodes are visited.
        """
        result: List[Any] = []

        def visit(node: TreeNode) -> None:
            result.append(node.value)
            for child in node.children:
                visit(child)

        visit(self.root)
        return result


## Demo Helpers

These helper functions run small examples for the selection algorithms and the data structures.

In [20]:
def demo_selection_algorithms() -> None:
    """Small demonstration for both selection methods."""
    print("=" * 70)
    print("SELECTION ALGORITHMS DEMO")
    print("=" * 70)

    sample = [9, 4, 7, 1, 3, 6, 8, 2, 5, 5, 3]
    k = 5

    print("Input array:", sample)
    print(f"Looking for the {k}-th smallest element")

    det_result = deterministic_select(sample, k)
    rand_result = randomized_select(sample, k)
    expected = sorted(sample)[k - 1]

    print("Deterministic select result:", det_result)
    print("Randomized select result:  ", rand_result)
    print("Sorted-array check:        ", expected)
    print("Correctness verified:      ", verify_selection_against_sort(sample, k))
    print()


def demo_data_structures() -> None:
    """Small demonstration for the elementary data structures."""
    print("=" * 70)
    print("DATA STRUCTURES DEMO")
    print("=" * 70)

    # ArrayList demo
    print("\nArrayList demo")
    array_list = ArrayList()
    array_list.append(10)
    array_list.append(20)
    array_list.append(30)
    array_list.insert(1, 15)
    removed = array_list.delete(2)
    print("After operations:", array_list)
    print("Removed value:", removed)
    print("Value at index 1:", array_list.get(1))

    # Matrix demo
    print("\nMatrix demo")
    matrix = Matrix(2, 2, fill=0)
    matrix.set(0, 1, 7)
    matrix.insert_row([8, 9])
    print(matrix)

    # Stack demo
    print("\nArrayStack demo")
    stack = ArrayStack()
    stack.push(1)
    stack.push(2)
    stack.push(3)
    print("Stack before pop:", stack)
    print("Popped:", stack.pop())
    print("Top now:", stack.peek())

    # Queue demo
    print("\nArrayQueue demo")
    queue = ArrayQueue()
    queue.enqueue("A")
    queue.enqueue("B")
    queue.enqueue("C")
    print("Queue before dequeue:", queue)
    print("Dequeued:", queue.dequeue())
    print("Front now:", queue.front())

    # Linked list demo
    print("\nSinglyLinkedList demo")
    linked = SinglyLinkedList()
    linked.insert_front(3)
    linked.insert_front(2)
    linked.insert_end(4)
    linked.insert_front(1)
    linked.delete_value(3)
    print(linked)
    print("Traversal:", linked.traverse())

    # Rooted tree demo
    print("\nRootedTree demo")
    tree = RootedTree("root")
    left = TreeNode("left")
    right = TreeNode("right")
    tree.root.add_child(left)
    tree.root.add_child(right)
    left.add_child(TreeNode("left.left"))
    left.add_child(TreeNode("left.right"))
    print("DFS order:", tree.dfs())
    print()


run_demo()




SELECTION ALGORITHMS DEMO
Input array: [9, 4, 7, 1, 3, 6, 8, 2, 5, 5, 3]
Looking for the 5-th smallest element
Deterministic select result: 4
Randomized select result:   4
Sorted-array check:         4
Correctness verified:       True

DATA STRUCTURES DEMO

ArrayList demo
After operations: ArrayList([10, 15, 30])
Removed value: 20
Value at index 1: 15

Matrix demo
Matrix(rows=3, cols=2, data=[[0, 7], [0, 0], [8, 9]])

ArrayStack demo
Stack before pop: ArrayStack([1, 2, 3])
Popped: 3
Top now: 2

ArrayQueue demo
Queue before dequeue: ArrayQueue(['A', 'B', 'C'])
Dequeued: A
Front now: B

SinglyLinkedList demo
SinglyLinkedList([1, 2, 4])
Traversal: [1, 2, 4]

RootedTree demo
DFS order: ['root', 'left', 'left.left', 'left.right', 'right']



## Run a quick demo

This cell runs both the selection and data-structure demos.

In [21]:
run_selection_benchmarks(output_file='selection_benchmark_results_colab.csv', trials=3)


Running selection benchmarks...

{'size': 1000, 'distribution': 'random', 'k': 500, 'deterministic_seconds': 0.0005064529999193231, 'randomized_seconds': 0.00022687766643988047}
{'size': 1000, 'distribution': 'sorted', 'k': 500, 'deterministic_seconds': 0.00035537466677245294, 'randomized_seconds': 0.00028339699989980244}
{'size': 1000, 'distribution': 'reverse_sorted', 'k': 500, 'deterministic_seconds': 0.00038889200004632585, 'randomized_seconds': 0.0001567089999904662}
{'size': 1000, 'distribution': 'many_duplicates', 'k': 500, 'deterministic_seconds': 0.00040636899999905535, 'randomized_seconds': 0.00020830766667738013}
{'size': 5000, 'distribution': 'random', 'k': 2500, 'deterministic_seconds': 0.0031638873330545416, 'randomized_seconds': 0.000951359999817214}
{'size': 5000, 'distribution': 'sorted', 'k': 2500, 'deterministic_seconds': 0.0020233076666045235, 'randomized_seconds': 0.0006895249998706277}
{'size': 5000, 'distribution': 'reverse_sorted', 'k': 2500, 'deterministic_seco

## Optional: quick correctness checks

This cell tests a few small cases, including duplicate-heavy arrays.

In [22]:

test_cases = [
    ([5], 1),
    ([3, 1, 2], 2),
    ([9, 4, 7, 1, 3, 6, 8, 2, 5], 4),
    ([5, 5, 5, 5, 5], 3),
    ([7, 2, 7, 1, 9, 1, 3], 5),
]

for arr, k in test_cases:
    expected = sorted(arr)[k - 1]
    det = deterministic_select(arr, k)
    rnd = randomized_select(arr, k)
    print(f"arr={arr}, k={k}")
    print(f"  expected      = {expected}")
    print(f"  deterministic = {det}")
    print(f"  randomized    = {rnd}")
    print()


arr=[5], k=1
  expected      = 5
  deterministic = 5
  randomized    = 5

arr=[3, 1, 2], k=2
  expected      = 2
  deterministic = 2
  randomized    = 2

arr=[9, 4, 7, 1, 3, 6, 8, 2, 5], k=4
  expected      = 4
  deterministic = 4
  randomized    = 4

arr=[5, 5, 5, 5, 5], k=3
  expected      = 5
  deterministic = 5
  randomized    = 5

arr=[7, 2, 7, 1, 9, 1, 3], k=5
  expected      = 7
  deterministic = 7
  randomized    = 7

